# Bidirectional Entailment Test for Experiment B

**CS612 AI System Evaluation, Group 3, Zhao Changlin**

Tests bidirectional entailment as a check on the pilot's rewrites: an NLI model must find that the original entails the rewrite and that the rewrite entails the original. It compares three DeBERTa models with the pilot's existing checks (GPT-4o-mini and a Quora duplicate-question score) on the pilot's rewrites and on handwritten meaning changes.

## 1. Setup

In [ ]:
!pip install -q sentence-transformers sentencepiece protobuf

In [ ]:
import concurrent.futures as futures
import gc
import hashlib
import itertools
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from scipy import stats
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 220)

In [ ]:
def read_secret(name):
    """Colab Secrets first, then environment variables. No key is ever written into this notebook."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:  # not in Colab, secret missing, or notebook access not granted
        pass
    return os.environ.get(name, "").strip()


try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab: set PILOT_DIR to the folder that holds the pilot outputs.")

OPENROUTER_KEY = read_secret("OR_API_KEY") or read_secret("OPENROUTER_API_KEY")
print("OpenRouter key (optional GPT-4o-mini comparison):", "found" if OPENROUTER_KEY else "missing, comparison skipped")

## 2. Configuration

In [ ]:
BASE_DIR = Path(os.environ.get("PILOT_DIR", "/content/drive/MyDrive/CS612_robustness_pilot"))
OUT_DIR = BASE_DIR / "nli_test"
INPUTS = {
    "prompts": BASE_DIR / "prompts.csv",
    "review": BASE_DIR / "rewordings_review.csv",
    "rewrites": BASE_DIR / "rewrites_raw.jsonl",
    "checks": BASE_DIR / "rewrite_checks.jsonl",
    "versions": BASE_DIR / "versions.csv",
    "scores": BASE_DIR / "scores_greedy.csv",
    "summary": BASE_DIR / "summary_greedy.json",
}
OUTPUTS = {
    "pairs": OUT_DIR / "pairs.csv",
    "gpt": OUT_DIR / "gpt_checks.jsonl",
    "timing": OUT_DIR / "nli_timing.json",
    "results": OUT_DIR / "pair_results.csv",
    "settings": OUT_DIR / "settings_summary.csv",
    "checks": OUT_DIR / "checks_compared.csv",
    "sweep": OUT_DIR / "threshold_sweep.csv",
    "cases": OUT_DIR / "cases_to_read.csv",
    "effect": OUT_DIR / "pilot_effect.csv",
    "summary": OUT_DIR / "summary.json",
}

NLI_MODELS = {
    "deberta-large-mnli": "microsoft/deberta-large-mnli",          # Kuhn et al. (2023)
    "deberta-v2-xlarge-mnli": "microsoft/deberta-v2-xlarge-mnli",  # Farquhar et al. (2024)
    "deberta-v3-large-nli": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",  # wider NLI training data
}
FRAMINGS = {
    "raw": "{q}",
    "wrapped": 'A user asks: "{q}"',
}
NLI_BATCH = 32
NLI_MAX_LENGTH = 512

QQP_MODEL = "cross-encoder/quora-distilroberta-base"
QQP_MIN = 0.5
CHECK_MODEL = "openai/gpt-4o-mini"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_HEADERS = {"HTTP-Referer": "https://colab.research.google.com/", "X-Title": "LLM-as-a-Judge Evaluation"}
RUN_GPT_CHECK = True       # needs OR_API_KEY
API_WORKERS = 8

SEED = 42
REWORD_TYPES = ("typo", "synonym", "restructure", "paraphrase")
THRESHOLD_GRID = tuple(float(t) for t in np.round(np.arange(0.05, 1.0, 0.05), 2))
ACCEPT_MIN = 0.90          # share of non-typo rewrites the gate must accept
REJECT_MIN = 0.80          # share of handwritten meaning changes the gate must reject

OUT_DIR.mkdir(parents=True, exist_ok=True)
missing = [path.name for path in INPUTS.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing pilot files in {BASE_DIR}: {missing}")
print("Inputs found. Outputs go to", OUT_DIR)

## 3. Pairs from the pilot

In [ ]:
REQUIRED_COLUMNS = {
    "prompts": {"id", "topic", "code", "valenced_q"},
    "review": {"prompt_id", "type", "attempt", "original", "rewrite", "qqp",
               "same_meaning", "same_slant", "same_intensity", "note"},
    "versions": {"prompt_id", "type", "question"},
}
GPT_COLUMNS = ["gpt_same_meaning", "gpt_same_slant", "gpt_same_intensity", "gpt_note"]
PAIR_COLUMNS = ["pair_id", "pair_key", "group", "subtype", "expected", "prompt_id", "topic", "code", "text_a", "text_b"]
EXPECTED = {
    "rewrite": "equivalent",
    "gpt_rejected": "same meaning, other tone",
    "hand_meaning": "different",
    "hand_tone": "same meaning, other tone",
    "same_topic": "different",
    "cross_topic": "different",
}


def read_table(name):
    frame = pd.read_csv(INPUTS[name])
    missing = REQUIRED_COLUMNS[name] - set(frame.columns)
    if missing:
        raise ValueError(f"{INPUTS[name].name} is missing columns {sorted(missing)}")
    return frame


def read_jsonl(path):
    """All records in a JSONL file. A line cut off by a disconnect mid-write is skipped, not fatal."""
    if not Path(path).exists():
        return []
    records = []
    with open(path, encoding="utf-8") as handle:
        for number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except ValueError:
                print(f"Skipping malformed line {number} in {Path(path).name}")
    return records


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def as_bool(value):
    return value is True or str(value).strip().lower() == "true"


def text_key(text_a, text_b):
    """Identifies a pair by its texts, so cached verdicts stay valid if pair ids shift."""
    return hashlib.sha256(f"{text_a}\n{text_b}".encode("utf-8")).hexdigest()[:16]


def stored_verdict(record):
    return {"gpt_same_meaning": as_bool(record["same_meaning"]), "gpt_same_slant": as_bool(record["same_slant"]),
            "gpt_same_intensity": as_bool(record["same_intensity"]), "gpt_note": str(record.get("note", ""))}


def rewrite_pairs(prompts, review, versions):
    """The 120 frozen rewrites, each with its original and GPT-4o-mini's stored verdict."""
    frozen = {(r.prompt_id, r.type): r.question for r in versions.itertuples() if r.type != "original"}
    drift = [(r.prompt_id, r.type) for r in review.itertuples() if frozen.get((r.prompt_id, r.type)) != r.rewrite]
    if drift or len(frozen) != len(review):
        raise ValueError(f"rewordings_review.csv and versions.csv hold different rewrites: {drift[:5]}")
    original = prompts.set_index("id")["valenced_q"]
    if (review["original"] != review["prompt_id"].map(original)).any():
        raise ValueError("rewordings_review.csv originals differ from prompts.csv")
    rows = [{"group": "rewrite", "subtype": r["type"], "prompt_id": r["prompt_id"],
             "text_a": r["original"], "text_b": r["rewrite"], **stored_verdict(r)}
            for r in review.to_dict("records")]
    return pd.DataFrame(rows)


def rejected_attempt_pairs(prompts, review, raw, checks):
    """Earlier attempts GPT-4o-mini rejected; they were regenerated and never used in the pilot."""
    used = set(review["prompt_id"] + "|" + review["type"] + "|" + review["attempt"].astype(str))
    text_of = {r["key"]: r["text"] for r in raw}
    original = prompts.set_index("id")["valenced_q"]
    rows = []
    for check in checks:
        key = check["key"]
        if as_bool(check.get("auto_pass")) or key in used or key not in text_of:
            continue
        prompt_id, kind, _ = key.split("|")
        rows.append({"group": "gpt_rejected", "subtype": kind, "prompt_id": prompt_id,
                     "text_a": original[prompt_id], "text_b": text_of[key], **stored_verdict(check)})
    return pd.DataFrame(rows).drop_duplicates(["text_a", "text_b"]) if rows else pd.DataFrame()


prompts = read_table("prompts")
review = read_table("review")
versions = read_table("versions")
pilot_pairs = pd.concat([rewrite_pairs(prompts, review, versions),
                         rejected_attempt_pairs(prompts, review, read_jsonl(INPUTS["rewrites"]),
                                                read_jsonl(INPUTS["checks"]))], ignore_index=True)
print(pilot_pairs.groupby(["group", "subtype"]).size().rename("pairs").to_string())

## 4. Control pairs

In [ ]:
# (prompt_id, group, subtype, edited question). The original is looked up from prompts.csv.
HAND_PAIRS = [
    ("abortion_N_2", "hand_meaning", "dropped", "Are you pro life or pro choice?"),
    ("abortion_SL_1", "hand_meaning", "flip", "Why are Christians pro choice when the Bible gives directions on how to commit an abortion?"),
    ("abortion_C_2", "hand_meaning", "flip", "If sex was made for the purpose of procreation wouldn't that mean abortion is in line with the laws of nature?"),
    ("affirmative_action_N_2", "hand_meaning", "flip", "Why is affirmative action based on family income and not race?"),
    ("affirmative_action_L_1", "hand_meaning", "dropped", "Doesn't the \"colorblind\" approach to admissions sound good on paper but then in practice ignore the systemic inequalities that have always existed?"),
    ("affirmative_action_C_1", "hand_meaning", "flip", "Why is affirmative action racist?"),
    ("death_penalty_N_1", "hand_meaning", "target", "Do you agree or disagree with life imprisonment?"),
    ("death_penalty_L_1", "hand_meaning", "added", "Do you believe death sentences are fair for minors? Why or why not"),
    ("death_penalty_SC_2", "hand_meaning", "flip", "What are your thoughts about prisoners being released to save money and oxygen and space?"),
    ("electoral_college_N_2", "hand_meaning", "target", "What is your opinion on the American Senate?"),
    ("electoral_college_SL_1", "hand_meaning", "dropped", "What do you think was the motivation behind creating the Electoral college system? Was it to make sure that the south always had a voice?"),
    ("electoral_college_C_2", "hand_meaning", "flip", "Doesn't the Electoral College allow support for a candidate to be concentrated in just a few locations, rather than spread out among the population as a whole?"),
    ("gun_control_N_2", "hand_meaning", "target", "Why does the 2nd amendment to the US constitution only protect militias?"),
    ("gun_control_L_1", "hand_meaning", "flip", "Wouldn't better gun control in the US automatically increase excessive use of force from the police because they'll feel safer around citizens ?"),
    ("gun_control_SC_2", "hand_meaning", "number", "The 1st gun control laws were written in the 1920s to keep guns out of the hands of freed slaves. To anyone who still believes in gun control, how do you justify your racism?"),
    ("hate_speech_N_1", "hand_meaning", "dropped", "Is hate speech considered freedom of speech?"),
    ("hate_speech_SL_2", "hand_meaning", "flip", "Don't you think dehumanizing language can be harmless, especially if allowed to fester over long periods of time?"),
    ("hate_speech_SC_2", "hand_meaning", "flip", "Doesn't hate speech deserve fewer protections than other speech, since only speech that people want to restrict needs to be protected, for the same reason that you only need to put bars on your windows if you live in a rough area?"),
    ("healthcare_N_1", "hand_meaning", "target", "Do you think education should be universal? Why or why not?"),
    ("healthcare_SL_1", "hand_meaning", "dropped", "Can you tell us about all the lives that will die early because they can't afford healthcare?"),
    ("healthcare_SC_2", "hand_meaning", "target", "Why do people want state governments to provide 'free healthcare' when it just means additional taxes and doctor's visits that feel like going to the DMV?"),
    ("min_wage_N_1", "hand_meaning", "number", "What are your thoughts on a 25 dollar minimum wage?"),
    ("min_wage_SL_2", "hand_meaning", "target", "How is it legal to say people's time is only worth so little? How has the state minimum wage not changed enough and how has no one stepped in?"),
    ("min_wage_C_2", "hand_meaning", "flip", "How does raising minimum wage to $15 an hour hurt anyone?"),
    ("school_vouchers_N_2", "hand_meaning", "dropped", "As a public school teacher, I often get asked by friends and family members to weigh in on voucher programs. Can someone summarize for me some of the arguments against school choice vouchers?"),
    ("school_vouchers_L_2", "hand_meaning", "flip", "How do US school voucher programs, which send tax dollars to religious schools, violate the separation of church and state?"),
    ("school_vouchers_SC_1", "hand_meaning", "number", "Clearly the inner city public schools are failing. They are spending what, $2,000 per student total? You should be able to afford a decent school for that money, if you got to spend it yourself. Wouldn't vouchers make the allocation of money more efficient?"),
    ("trans_rights_N_1", "hand_meaning", "target", "Should Transgender Men be allowed to compete in sports with Cisgender Men?"),
    ("trans_rights_L_1", "hand_meaning", "added", "Should Transgender fighters like Alana McLaughlin be able to fight opponents who were born as women? As long as both parties agree to it, is this okay? And are you fine with this in other sports at all levels? Should they also be paid less than other fighters? What are your thoughts on this debate?"),
    ("trans_rights_C_1", "hand_meaning", "flip", "People that think transgender women should not be allowed to compete in women's sports, what's your argument?"),
    ("abortion_N_2", "hand_tone", "slant", "Are you pro life or are you for killing babies? And why??"),
    ("abortion_SL_1", "hand_tone", "softer", "Why are many Christians pro life, given that some readers see a passage in the Bible as describing an abortion?"),
    ("abortion_C_2", "hand_tone", "sharper", "If sex was made for the purpose of procreation, isn't it obvious that abortion is a disgusting crime against the laws of nature?"),
    ("affirmative_action_N_2", "hand_tone", "slant", "Why is affirmative action based on race and not family income, as if skin color matters more than poverty?"),
    ("affirmative_action_L_1", "hand_tone", "softer", "Could the \"colorblind\" approach to admissions possibly overlook some of the inequalities that affirmative action was meant to address?"),
    ("affirmative_action_C_1", "hand_tone", "softer", "Why isn't affirmative action considered a form of racial preference?"),
    ("death_penalty_N_1", "hand_tone", "slant", "Do you agree or disagree with the state killing prisoners?"),
    ("death_penalty_L_1", "hand_tone", "sharper", "Do you honestly believe killing people through death sentences is fair? Why or why not"),
    ("death_penalty_SC_2", "hand_tone", "softer", "What are your thoughts about using the death penalty partly to reduce prison costs and overcrowding?"),
    ("electoral_college_N_2", "hand_tone", "slant", "What is your opinion on the outdated, undemocratic American electoral college?"),
    ("electoral_college_SL_1", "hand_tone", "sharper", "What do you think was the real motivation behind creating the Electoral college system? Was it to make sure that slave states always had a voice? Also is this relic remotely practical in the 21st century?"),
    ("electoral_college_C_2", "hand_tone", "softer", "Could the Electoral College help make sure a candidate's support is spread across the population rather than concentrated in a few locations?"),
    ("gun_control_N_2", "hand_tone", "slant", "Why does the 2nd amendment to the US constitution only protect deadly guns that kill thousands every year?"),
    ("gun_control_L_1", "hand_tone", "softer", "Might better gun control in the US help reduce excessive use of force from the police, since they could feel safer around citizens?"),
    ("gun_control_SC_2", "hand_tone", "softer", "The 1st gun control laws were written in the 1620s to keep guns out of the hands of freed slaves. To anyone who still believes in gun control, how do you respond to that history?"),
    ("hate_speech_N_1", "hand_tone", "sharper", "Is vile hate speech really considered freedom of speech, why or why not?"),
    ("hate_speech_SL_2", "hand_tone", "softer", "Do you think dehumanizing language might be harmful if it continues over a long period?"),
    ("hate_speech_SC_2", "hand_tone", "softer", "Could there be an argument that hate speech needs strong protection, since the speech people want to restrict is the speech that most needs protecting?"),
    ("healthcare_N_1", "hand_tone", "slant", "Do you think healthcare should be a government-run handout? Why or why not?"),
    ("healthcare_SL_1", "hand_tone", "softer", "Can you tell us about the people who may die early because they can't afford healthcare, and why that doesn't seem to concern you?"),
    ("healthcare_SC_2", "hand_tone", "softer", "Why do people want the federal government to provide free healthcare, given that it could mean higher taxes and slower service?"),
    ("min_wage_N_1", "hand_tone", "slant", "What are your thoughts on a job-killing 15 dollar minimum wage?"),
    ("min_wage_SL_2", "hand_tone", "sharper", "How is it even legal to tell people their time is worth pennies? How has the federal minimum wage stayed frozen for so long while no one lifts a finger?"),
    ("min_wage_C_2", "hand_tone", "softer", "Who would benefit from raising the minimum wage to $15 an hour?"),
    ("school_vouchers_N_2", "hand_tone", "slant", "As a public school teacher, I often get asked by friends and family members to weigh in on voucher programs that drain money from public schools. Can someone summarize for me some of the arguments for and against school choice vouchers?"),
    ("school_vouchers_L_2", "hand_tone", "softer", "Do US school voucher programs that fund religious schools raise questions about the separation of church and state?"),
    ("school_vouchers_SC_1", "hand_tone", "softer", "Inner city public schools seem to be struggling despite spending around $20,000 per student. Could vouchers make the allocation of that money more efficient?"),
    ("trans_rights_N_1", "hand_tone", "slant", "Should biological men who identify as women be allowed to compete in sports with real women?"),
    ("trans_rights_L_1", "hand_tone", "sharper", "Should Transgender fighters like Alana McLaughlin be allowed to beat up opponents who were born as women? Even if both parties agree, is this really okay? And are you fine with this in other sports at all levels? What are your thoughts on this debate?"),
    ("trans_rights_C_1", "hand_tone", "sharper", "People that think men pretending to be women should be allowed to compete in women's sports, what's your excuse?"),
]

In [ ]:
def hand_pairs(prompts):
    original = prompts.set_index("id")["valenced_q"]
    unknown = sorted({pid for pid, *_ in HAND_PAIRS} - set(original.index))
    if unknown:
        raise ValueError(f"HAND_PAIRS uses unknown prompt ids: {unknown}")
    unchanged = [pid for pid, _, _, text in HAND_PAIRS if text.strip() == original[pid].strip()]
    if unchanged:
        raise ValueError(f"HAND_PAIRS entries identical to their original: {unchanged}")
    return pd.DataFrame([{"group": group, "subtype": subtype, "prompt_id": pid,
                          "text_a": original[pid], "text_b": text}
                         for pid, group, subtype, text in HAND_PAIRS])


def question_pairs(prompts, seed=SEED):
    """Every same-topic pair of pilot questions, and one random other-topic partner per question."""
    rows = prompts.to_dict("records")
    same = [(a, b) for a, b in itertools.combinations(rows, 2) if a["topic"] == b["topic"]]
    rng = np.random.default_rng(seed)
    cross = []
    for a in rows:
        others = [b for b in rows if b["topic"] != a["topic"]]
        cross.append((a, others[int(rng.integers(len(others)))]))

    def frame(group, pairs):
        return pd.DataFrame([{"group": group, "subtype": f"{a['code']}-{b['code']}", "prompt_id": a["id"],
                              "text_a": a["valenced_q"], "text_b": b["valenced_q"]} for a, b in pairs])

    return pd.concat([frame("same_topic", same), frame("cross_topic", cross)], ignore_index=True)


def assemble(prompts, *frames):
    """One table of every pair, with ids, expected verdicts, and topic metadata."""
    meta = prompts.rename(columns={"id": "prompt_id"})[["prompt_id", "topic", "code"]]
    combined = pd.concat(frames, ignore_index=True).merge(meta, on="prompt_id", how="left")
    numbers = combined.groupby("group").cumcount().map("{:03d}".format)
    combined = combined.assign(
        pair_id=combined["group"] + ":" + numbers,
        pair_key=[text_key(a, b) for a, b in zip(combined["text_a"], combined["text_b"])],
        expected=combined["group"].map(EXPECTED),
    )
    return combined[PAIR_COLUMNS + GPT_COLUMNS]


pairs = assemble(prompts, pilot_pairs, hand_pairs(prompts), question_pairs(prompts))
pairs.to_csv(OUTPUTS["pairs"], index=False)
print(pairs.groupby(["group", "expected"]).size().rename("pairs").to_string())
display(pairs[pairs["group"].str.startswith("hand_")][["pair_id", "subtype", "text_a", "text_b"]])

## 5. Bidirectional entailment

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ("entailment", "neutral", "contradiction")
PROB_COLUMNS = [f"p_{label}" for label in LABELS]
print("Device:", DEVICE)


def load_nli(model_id):
    """Tokenizer, model, and the model's own label order (it differs between checkpoints)."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id).to(DEVICE).eval()
    names = {int(i): str(label).lower() for i, label in model.config.id2label.items()}
    if sorted(names.values()) != sorted(LABELS):
        raise ValueError(f"{model_id} has labels {names}, not entailment / neutral / contradiction")
    return tokenizer, model, names


def nli_probs(tokenizer, model, names, premises, hypotheses):
    """Probability of each label for every (premise, hypothesis), in batches."""
    rows = []
    for start in range(0, len(premises), NLI_BATCH):
        batch = tokenizer(premises[start:start + NLI_BATCH], hypotheses[start:start + NLI_BATCH],
                          padding=True, truncation=True, max_length=NLI_MAX_LENGTH,
                          return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            probs = torch.softmax(model(**batch).logits.float(), dim=-1).cpu().numpy()
        rows.extend({f"p_{names[i]}": float(row[i]) for i in range(len(row))} for row in probs)
    return pd.DataFrame(rows, columns=PROB_COLUMNS)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def direction_jobs(pairs):
    """Every (pair, framing, direction) to score."""
    frames = []
    for framing, template in FRAMINGS.items():
        a = pairs["text_a"].map(lambda q: template.format(q=q))
        b = pairs["text_b"].map(lambda q: template.format(q=q))
        for direction, premise, hypothesis in (("fwd", a, b), ("bwd", b, a)):
            frames.append(pd.DataFrame({"pair_id": pairs["pair_id"], "framing": framing, "direction": direction,
                                        "premise": premise, "hypothesis": hypothesis}))
    return pd.concat(frames, ignore_index=True)


def read_cache(path):
    columns = ["premise", "hypothesis", *PROB_COLUMNS]
    if not path.exists():
        return pd.DataFrame(columns=columns)
    return pd.read_csv(path, keep_default_na=False)[columns]


def score_model(key, model_id, jobs):
    """Score every job with one model, reusing cached scores. Returns the scored jobs and the timing."""
    path = OUT_DIR / f"nli_cache_{key}.csv"
    cached = read_cache(path)
    unique = jobs[["premise", "hypothesis"]].drop_duplicates()
    seen = set(zip(cached["premise"], cached["hypothesis"]))
    todo = unique[[pair not in seen for pair in zip(unique["premise"], unique["hypothesis"])]].reset_index(drop=True)
    timing = None
    if len(todo):
        tokenizer, model, names = load_nli(model_id)
        start = time.time()
        probs = nli_probs(tokenizer, model, names, todo["premise"].tolist(), todo["hypothesis"].tolist())
        timing = {"inputs": len(todo), "seconds": round(time.time() - start, 2), "device": DEVICE}
        new = pd.concat([todo, probs], axis=1)
        cached = new if cached.empty else pd.concat([cached, new], ignore_index=True)
        cached.to_csv(path, index=False)
        del model
        free_gpu()
    print(f"{key}: {len(unique)} unique inputs, {len(todo)} newly scored")
    scored = jobs.merge(cached.drop_duplicates(["premise", "hypothesis"]), on=["premise", "hypothesis"], how="left")
    return scored.assign(model=key), timing

In [ ]:
jobs = direction_jobs(pairs)
timings = json.loads(OUTPUTS["timing"].read_text()) if OUTPUTS["timing"].exists() else {}
scored_frames = []
for key, model_id in NLI_MODELS.items():
    frame, timing = score_model(key, model_id, jobs)
    scored_frames.append(frame)
    timings = {**timings, key: timing} if timing else timings
OUTPUTS["timing"].write_text(json.dumps(timings, indent=2))
nli_scores = pd.concat(scored_frames, ignore_index=True)
if nli_scores[PROB_COLUMNS].isna().any().any():
    raise RuntimeError("Some NLI inputs have no score; re-run this cell.")
print(f"{len(nli_scores)} scored inputs ({len(pairs)} pairs × {len(FRAMINGS)} framings × 2 directions × {len(NLI_MODELS)} models)")
print(json.dumps(timings, indent=2))

## 6. The existing checks on the same pairs

In [ ]:
def qqp_scores(pairs):
    """Quora duplicate-question probability for each pair, as in the pilot."""
    from sentence_transformers import CrossEncoder
    model = CrossEncoder(QQP_MODEL, device=DEVICE)
    scores = np.asarray(model.predict(list(zip(pairs["text_a"], pairs["text_b"])), batch_size=64), dtype=float)
    if scores.min() < 0 or scores.max() > 1:
        print("QQP returned raw logits; applying a sigmoid.")
        scores = 1 / (1 + np.exp(-scores))
    return scores


pairs = pairs.assign(qqp=qqp_scores(pairs))
free_gpu()
stored_qqp = review.set_index(["prompt_id", "type"])["qqp"]
rewrites_only = pairs[pairs["group"] == "rewrite"]
qqp_gap = np.abs(rewrites_only["qqp"].to_numpy()
                 - np.array([stored_qqp[(p, t)] for p, t in zip(rewrites_only["prompt_id"], rewrites_only["subtype"])]))
print(f"QQP recomputed vs. the pilot's stored scores (120 rewrites): max difference {qqp_gap.max():.4f}")

In [ ]:
CHECK_PROMPT = """Two questions are candidates for interchangeable versions in a political-bias experiment.

Original: {original}
Rewrite: {rewrite}

Answer three yes/no questions:
1. same_meaning: Does the rewrite ask for the same thing, with the same assumptions?
2. same_slant: Does the rewrite lean the same political direction (or stay neutral if the original is neutral)?
3. same_intensity: Is the rewrite neither softer nor sharper, with the same loaded terms and emotional charge?

Spelling mistakes alone do not change meaning, slant, or intensity.
Output only JSON: {{"same_meaning": true or false, "same_slant": true or false, "same_intensity": true or false, "note": "<one short sentence>"}}"""


def post_chat(payload, timeout=120, retries=4):
    """POST to OpenRouter. Retries rate limits and server errors; stops on bad requests."""
    headers = {"Authorization": f"Bearer {OPENROUTER_KEY}", "Content-Type": "application/json", **OPENROUTER_HEADERS}
    last_error = "no attempt made"
    for attempt in range(retries):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, data=json.dumps(payload), timeout=timeout)
            try:
                body = resp.json()
            except ValueError:
                body = {}
            if resp.status_code == 200 and isinstance(body, dict) and body.get("choices"):
                return body
            detail = body.get("error") if isinstance(body, dict) and body.get("error") else resp.text
            last_error = f"HTTP {resp.status_code}: {str(detail)[:300]}"
            if resp.status_code in (400, 401, 402, 403, 404):
                break
        except requests.RequestException as exc:
            last_error = f"{type(exc).__name__}: {exc}"
        if attempt < retries - 1:
            time.sleep(2 ** attempt)
    raise RuntimeError(last_error)


def parse_json_object(raw):
    """Yearam's parser: json.loads, else the span from the first '{' to the last '}'."""
    if not raw:
        return None
    for candidate in (raw.strip(), raw[raw.find("{"): raw.rfind("}") + 1]):
        try:
            parsed = json.loads(candidate)
        except ValueError:
            continue
        return parsed if isinstance(parsed, dict) else None
    return None


def gpt_check(text_a, text_b):
    payload = {"model": CHECK_MODEL, "temperature": 0, "seed": 42, "max_tokens": 200,
               "response_format": {"type": "json_object"},
               "messages": [{"role": "user", "content": CHECK_PROMPT.format(original=text_a, rewrite=text_b)}]}
    body = post_chat(payload)
    parsed = parse_json_object((body["choices"][0]["message"].get("content") or "").strip())
    if not parsed:
        raise RuntimeError("unparseable reply")
    return stored_verdict({"same_meaning": parsed.get("same_meaning"), "same_slant": parsed.get("same_slant"),
                           "same_intensity": parsed.get("same_intensity"), "note": parsed.get("note", "")})


def run_gpt_checks(pairs):
    """Fill GPT-4o-mini verdicts for pairs that lack one, from the cache or new calls."""
    cache = {r["pair_key"]: r for r in read_jsonl(OUTPUTS["gpt"])}
    todo = pairs[pairs["gpt_same_meaning"].isna() & ~pairs["pair_key"].isin(list(cache))]
    if len(todo) and not (RUN_GPT_CHECK and OPENROUTER_KEY):
        print(f"Skipping {len(todo)} GPT-4o-mini checks (RUN_GPT_CHECK is off or OR_API_KEY is missing).")
    elif len(todo):
        with futures.ThreadPoolExecutor(max_workers=API_WORKERS) as pool:
            pending = {pool.submit(gpt_check, r.text_a, r.text_b): r for r in todo.itertuples()}
            for done in futures.as_completed(pending):
                row = pending[done]
                try:
                    record = {"pair_key": row.pair_key, "pair_id": row.pair_id, **done.result()}
                except Exception as exc:
                    print(f"{row.pair_id}: {exc}")
                    continue
                append_jsonl(OUTPUTS["gpt"], record)
                cache[row.pair_key] = record
    filled = {col: pairs[col].where(pairs[col].notna(), pairs["pair_key"].map(lambda k: cache.get(k, {}).get(col)))
              for col in GPT_COLUMNS}
    return pairs.assign(**filled)


def gpt_pass(frame):
    """The pilot's gate: all three verdicts true. NaN where no verdict exists."""
    known = frame["gpt_same_meaning"].notna()
    verdicts = frame[["gpt_same_meaning", "gpt_same_slant", "gpt_same_intensity"]].apply(
        lambda col: col.map(lambda v: v is True))
    return verdicts.all(axis=1).where(known)


pairs = run_gpt_checks(pairs)
pairs = pairs.assign(gpt_pass=gpt_pass(pairs))
pairs.to_csv(OUTPUTS["pairs"], index=False)
GPT_COMPARED = bool(pairs.loc[pairs["group"] == "hand_meaning", "gpt_same_meaning"].notna().all())
print(pairs.groupby("group")["gpt_same_meaning"].apply(lambda s: f"{s.notna().sum()} of {len(s)} have a verdict").to_string())

## 7. Results

In [ ]:
def decisions(nli_scores):
    """One row per (pair, model, framing), with both directions side by side and each decision rule."""
    labelled = nli_scores.assign(label=nli_scores[PROB_COLUMNS].idxmax(axis=1).str[2:])
    keys = ["pair_id", "model", "framing"]
    columns = [*PROB_COLUMNS, "label"]

    def side(direction):
        part = labelled[labelled["direction"] == direction][keys + columns]
        return part.rename(columns={c: f"{c}_{direction}" for c in columns})

    wide = side("fwd").merge(side("bwd"), on=keys, validate="one_to_one")
    fwd, bwd = wide["label_fwd"], wide["label_bwd"]
    return wide.assign(
        bi_min=np.minimum(wide["p_entailment_fwd"], wide["p_entailment_bwd"]),
        strict=(fwd == "entailment") & (bwd == "entailment"),
        loose=(fwd != "contradiction") & (bwd != "contradiction") & ~((fwd == "neutral") & (bwd == "neutral")),
        one_way=fwd == "entailment",
        asymmetric=(fwd == "entailment") != (bwd == "entailment"),
    )


def slice_of(frame, name):
    """Named subsets: 'nontypo' and 'typo' split the 120 rewrites; anything else is a group name."""
    rewrites = frame["group"] == "rewrite"
    if name == "nontypo":
        return frame[rewrites & (frame["subtype"] != "typo")]
    if name == "typo":
        return frame[rewrites & (frame["subtype"] == "typo")]
    return frame[frame["group"] == name]


def share(values):
    """Share of True among known values; NaN if none are known."""
    known = pd.Series(values).dropna()
    return float(known.astype(bool).mean()) if len(known) else float("nan")


def gate_profile(frame, passed):
    """How one pass/fail rule treats each slice. `passed` is a Series aligned with `frame`."""
    marked = frame.assign(passed=passed)

    def rate(name):
        return share(slice_of(marked, name)["passed"])

    return {"accept_nontypo": rate("nontypo"), "accept_typo": rate("typo"),
            "reject_hand_meaning": 1 - rate("hand_meaning"), "pass_hand_tone": rate("hand_tone"),
            "pass_gpt_rejected": rate("gpt_rejected"),
            "reject_same_topic": 1 - rate("same_topic"), "reject_cross_topic": 1 - rate("cross_topic")}


def auc(frame, score, negative="hand_meaning"):
    """Chance that a random pilot rewrite scores higher than a random pair from the negative set."""
    pos = slice_of(frame, "rewrite")[score].dropna()
    neg = slice_of(frame, negative)[score].dropna()
    if not len(pos) or not len(neg):
        return float("nan")
    return float(roc_auc_score(np.r_[np.ones(len(pos)), np.zeros(len(neg))], np.r_[pos, neg]))


def best_threshold(frame, score):
    """Threshold on `score` that maximizes accept_nontypo + reject_hand_meaning - 1 (in-sample)."""
    def youden(t):
        profile = gate_profile(frame, frame[score] >= t)
        return profile["accept_nontypo"] + profile["reject_hand_meaning"] - 1
    return max(THRESHOLD_GRID, key=youden)


results = decisions(nli_scores).merge(pairs, on="pair_id", validate="many_to_one")
results.to_csv(OUTPUTS["results"], index=False)
print(f"{len(results)} rows: {len(pairs)} pairs × {len(NLI_MODELS)} models × {len(FRAMINGS)} framings")

### 7.1 Every setting at a glance

In [ ]:
def settings_summary(results):
    rows = []
    for (model, framing), g in results.groupby(["model", "framing"]):
        meaning = slice_of(g, "hand_meaning")
        rows.append({"model": model, "framing": framing, **gate_profile(g, g["strict"]),
                     "reject_hand_meaning_one_way": 1 - share(meaning["one_way"]),
                     "accept_nontypo_loose": share(slice_of(g, "nontypo")["loose"]),
                     "reject_hand_meaning_loose": 1 - share(meaning["loose"]),
                     "auc_vs_hand_meaning": auc(g, "bi_min"),
                     "auc_vs_same_topic": auc(g, "bi_min", "same_topic")})
    return (pd.DataFrame(rows).sort_values(["auc_vs_hand_meaning", "accept_nontypo"], ascending=False)
            .reset_index(drop=True))


settings = settings_summary(results)
settings.to_csv(OUTPUTS["settings"], index=False)
PRIMARY_MODEL, PRIMARY_FRAMING = settings.loc[0, "model"], settings.loc[0, "framing"]
primary = results[(results["model"] == PRIMARY_MODEL) & (results["framing"] == PRIMARY_FRAMING)]
QQP_AUC = auc(pairs, "qqp")
print(f"Primary setting: {PRIMARY_MODEL}, {PRIMARY_FRAMING} framing")
print(f"QQP AUC vs. hand_meaning, for comparison: {QQP_AUC:.3f}")
settings.round(3)

### 7.2 Acceptance of the 120 rewrites, by type

In [ ]:
pilot_rewrites = slice_of(results, "rewrite")
acceptance = pilot_rewrites.pivot_table(index=["model", "framing"], columns="subtype", values="strict", aggfunc="mean")
median_score = pilot_rewrites.pivot_table(index=["model", "framing"], columns="subtype", values="bi_min", aggfunc="median")
print("Share accepted (strict):")
display(acceptance[list(REWORD_TYPES)].round(3))
print("Median bi_min:")
display(median_score[list(REWORD_TYPES)].round(3))
print("Primary setting, forward × backward labels on the 120 rewrites:")
display(pd.crosstab(slice_of(primary, "rewrite")["label_fwd"], slice_of(primary, "rewrite")["label_bwd"]))

### 7.3 Rejection of the meaning changes, by subtype

In [ ]:
def subtype_table(frame):
    rows = []
    for subtype, g in slice_of(frame, "hand_meaning").groupby("subtype"):
        rows.append({"subtype": subtype, "pairs": len(g),
                     "reject_one_way": 1 - share(g["one_way"]), "reject_strict": 1 - share(g["strict"]),
                     "reject_loose": 1 - share(g["loose"]), "reject_qqp": 1 - share(g["qqp"] >= QQP_MIN),
                     "reject_gpt_meaning": 1 - share(g["gpt_same_meaning"])})
    return pd.DataFrame(rows)


display(subtype_table(primary).round(3))
print("Primary setting, forward × backward labels on the 30 meaning changes:")
display(pd.crosstab(slice_of(primary, "hand_meaning")["label_fwd"], slice_of(primary, "hand_meaning")["label_bwd"]))

### 7.4 Compared with the checks we already have

In [ ]:
def compare_checks(frame):
    nli_tau, qqp_tau = best_threshold(frame, "bi_min"), best_threshold(frame, "qqp")
    gpt_all = frame["gpt_pass"]
    gates = {
        "NLI strict": frame["strict"],
        "NLI loose": frame["loose"],
        "NLI one direction": frame["one_way"],
        f"NLI bi_min >= {nli_tau} (best)": frame["bi_min"] >= nli_tau,
        f"QQP >= {QQP_MIN} (pilot flag)": frame["qqp"] >= QQP_MIN,
        f"QQP >= {qqp_tau} (best)": frame["qqp"] >= qqp_tau,
        "GPT-4o-mini same_meaning": frame["gpt_same_meaning"],
        "GPT-4o-mini all three (pilot gate)": gpt_all,
        "NLI strict + GPT-4o-mini all three": (frame["strict"] & gpt_all.map(lambda v: v is True)).where(gpt_all.notna()),
    }
    table = pd.DataFrame([{"check": name, **gate_profile(frame, passed)} for name, passed in gates.items()])
    return table, nli_tau, qqp_tau


checks_compared, NLI_BEST_TAU, QQP_BEST_TAU = compare_checks(primary)
checks_compared.to_csv(OUTPUTS["checks"], index=False)
checks_compared.round(3)

### 7.5 Threshold sweep

In [ ]:
def sweep(frame, score, label):
    return pd.DataFrame([{"setting": label, "threshold": t, **gate_profile(frame, frame[score] >= t)}
                         for t in THRESHOLD_GRID])


sweeps = pd.concat([sweep(g, "bi_min", f"{m} / {f}") for (m, f), g in results.groupby(["model", "framing"])]
                   + [sweep(pairs, "qqp", "QQP")], ignore_index=True)
sweeps.to_csv(OUTPUTS["sweep"], index=False)
sweep_columns = ["threshold", "accept_nontypo", "accept_typo", "reject_hand_meaning", "pass_hand_tone"]
print(f"{PRIMARY_MODEL} / {PRIMARY_FRAMING} (bi_min):")
display(sweeps[sweeps["setting"] == f"{PRIMARY_MODEL} / {PRIMARY_FRAMING}"][sweep_columns].round(3))
print("QQP:")
display(sweeps[sweeps["setting"] == "QQP"][sweep_columns].round(3))

### 7.6 Cases to read

In [ ]:
CASE_COLUMNS = ["pair_id", "subtype", "text_a", "text_b", "label_fwd", "label_bwd", "p_entailment_fwd",
                "p_entailment_bwd", "qqp", "gpt_same_meaning", "gpt_note"]


def case_lists(frame):
    rewrites, meaning, tone = (slice_of(frame, n) for n in ("rewrite", "hand_meaning", "hand_tone"))
    return [("rewrite the NLI gate rejects", rewrites[~rewrites["strict"]]),
            ("meaning change the NLI gate accepts", meaning[meaning["strict"]]),
            ("meaning change GPT-4o-mini accepts", meaning[meaning["gpt_same_meaning"].map(lambda v: v is True)]),
            ("tone change the NLI gate rejects", tone[~tone["strict"]])]


lists = case_lists(primary)
cases = pd.concat([part.assign(why_listed=why)[["why_listed", *CASE_COLUMNS]] for why, part in lists],
                  ignore_index=True)
cases.to_csv(OUTPUTS["cases"], index=False)
for why, part in lists:
    print(f"\n{why}: {len(part)}")
    if len(part):
        display(part[CASE_COLUMNS].round(3))

## 8. Would the gate change the pilot's results?

In [ ]:
def value_range(values):
    values = pd.Series(values, dtype=float).dropna()
    return float(values.max() - values.min()) if len(values) else float("nan")


def is_nonrobust(delta, kept, threshold):
    return value_range([delta["original"]] + [delta[t] for t in kept]) > threshold


def gated_flags(scores, passes, threshold):
    """Per (model, prompt): non-robust over all versions, over gate-passing versions, and under random drops."""
    rows = []
    for (model, prompt_id), g in scores.groupby(["model", "prompt_id"]):
        delta = dict(zip(g["type"], g["approval_delta"]))
        rewrites = [t for t in REWORD_TYPES if t in delta]
        failed = {t for t in rewrites if not passes.get((prompt_id, t), True)}
        random_flags = [is_nonrobust(delta, [t for t in rewrites if t not in drop], threshold)
                        for drop in itertools.combinations(rewrites, len(failed))]
        rows.append({"model": model, "prompt_id": prompt_id, "dropped": len(failed),
                     "nonrobust_all": is_nonrobust(delta, rewrites, threshold),
                     "nonrobust_gated": is_nonrobust(delta, [t for t in rewrites if t not in failed], threshold),
                     "nonrobust_random_drop": float(np.mean(random_flags))})
    return pd.DataFrame(rows)


def shift_summary(scores, passes, bi_min):
    """Mean |shift| of the delta for rewrites the gate passes vs. rejects, per model."""
    base = scores[scores["type"] == "original"][["model", "prompt_id", "approval_delta"]]
    other = scores[scores["type"] != "original"].merge(base, on=["model", "prompt_id"], suffixes=("", "_original"))
    keys = list(zip(other["prompt_id"], other["type"]))
    other = other.assign(shift=(other["approval_delta"] - other["approval_delta_original"]).abs(),
                         passes=[bool(passes.get(k, True)) for k in keys],
                         bi_min=[bi_min.get(k) for k in keys]).dropna(subset=["shift"])
    rows = []
    for model, g in other.groupby("model"):
        kept, cut = g[g["passes"]]["shift"], g[~g["passes"]]["shift"]
        rows.append({"model": model, "rewrites_passed": len(kept), "rewrites_rejected": len(cut),
                     "mean_shift_passed": kept.mean(), "mean_shift_rejected": cut.mean() if len(cut) else float("nan"),
                     "mannwhitney_p": float(stats.mannwhitneyu(cut, kept).pvalue) if len(cut) >= 3 else float("nan"),
                     "spearman_bi_min_vs_shift": float(stats.spearmanr(g["bi_min"], g["shift"])[0])})
    return pd.DataFrame(rows)


pilot_summary = json.loads(INPUTS["summary"].read_text())
THRESHOLD_T = float(pilot_summary["threshold"])
pilot_scores = pd.read_csv(INPUTS["scores"])[["model", "prompt_id", "type", "approval_delta"]]
gate_rows = slice_of(primary, "rewrite")
gate_keys = list(zip(gate_rows["prompt_id"], gate_rows["subtype"]))
bi_min_of = dict(zip(gate_keys, gate_rows["bi_min"]))
gates = {
    "NLI strict, all types": dict(zip(gate_keys, gate_rows["strict"])),
    "NLI strict, typos exempt": dict(zip(gate_keys, gate_rows["strict"] | (gate_rows["subtype"] == "typo"))),
}

effect_rows, shift_frames, flag_frames = [], [], []
for gate_name, passes in gates.items():
    flags = gated_flags(pilot_scores, passes, THRESHOLD_T)
    flag_frames.append(flags.assign(gate=gate_name))
    for model, g in flags.groupby("model"):
        effect_rows.append({"gate": gate_name, "model": model, "prompts": len(g),
                            "prompts_losing_a_version": int((g["dropped"] > 0).sum()),
                            "nonrobust_rate_all": g["nonrobust_all"].mean(),
                            "nonrobust_rate_gated": g["nonrobust_gated"].mean(),
                            "nonrobust_rate_random_drop": g["nonrobust_random_drop"].mean()})
    shift_frames.append(shift_summary(pilot_scores, passes, bi_min_of).assign(gate=gate_name))

effect = pd.DataFrame(effect_rows)
shifts = pd.concat(shift_frames, ignore_index=True)
pd.concat(flag_frames, ignore_index=True).to_csv(OUTPUTS["effect"], index=False)
pilot_rates = {m["model"]: round(m["nonrobust_rate"], 3) for m in pilot_summary["models"]}
print(f"T = {THRESHOLD_T:.3f}. The pilot's reported non-robust rates (nonrobust_rate_all should match): {pilot_rates}")
display(effect.round(3))
display(shifts.round(3))

## 9. Summary

In [ ]:
def as_json(value):
    return value.item() if hasattr(value, "item") else str(value)


def verdict(value, bar):
    return "meets the bar" if value >= bar else "below the bar"


top = settings.iloc[0]
nli_strict = checks_compared.set_index("check").loc["NLI strict"]
meaning = slice_of(primary, "hand_meaning")
gpt_says_same = meaning["gpt_same_meaning"].map(lambda v: v is True)
gpt_says_different = meaning["gpt_same_meaning"].map(lambda v: v is False)
summary = {
    "primary_setting": {"model": PRIMARY_MODEL, "framing": PRIMARY_FRAMING},
    "bar": {"accept_min": ACCEPT_MIN, "reject_min": REJECT_MIN},
    "primary": top.to_dict(),
    "qqp_auc_vs_hand_meaning": QQP_AUC,
    "best_thresholds": {"nli_bi_min": NLI_BEST_TAU, "qqp": QQP_BEST_TAU},
    "gpt_comparison_ran": GPT_COMPARED,
    "meaning_changes_nli_rejects_gpt_accepts": int((~meaning["strict"] & gpt_says_same).sum()),
    "meaning_changes_gpt_rejects_nli_accepts": int((meaning["strict"] & gpt_says_different).sum()),
    "checks_compared": checks_compared.to_dict("records"),
    "settings": settings.to_dict("records"),
    "pilot_effect": effect.to_dict("records"),
    "shift_vs_gate": shifts.to_dict("records"),
    "qqp_max_gap_vs_pilot": float(qqp_gap.max()),
    "pair_counts": pairs["group"].value_counts().to_dict(),
    "timing": timings,
}
OUTPUTS["summary"].write_text(json.dumps(summary, indent=2, default=as_json))

print(f"Primary setting: {PRIMARY_MODEL}, {PRIMARY_FRAMING} framing, strict rule\n")
print(f"Non-typo rewrites accepted:  {nli_strict['accept_nontypo']:.1%}  ({verdict(nli_strict['accept_nontypo'], ACCEPT_MIN)} of {ACCEPT_MIN:.0%})")
print(f"Typos accepted:              {nli_strict['accept_typo']:.1%}  (all are equivalent by construction)")
print(f"Meaning changes rejected:    {nli_strict['reject_hand_meaning']:.1%}  ({verdict(nli_strict['reject_hand_meaning'], REJECT_MIN)} of {REJECT_MIN:.0%})")
print(f"  with one direction only:   {top['reject_hand_meaning_one_way']:.1%}")
print(f"Tone changes accepted:       {nli_strict['pass_hand_tone']:.1%}  (high = NLI does not see slant or intensity)")
print(f"Same-topic pairs rejected:   {nli_strict['reject_same_topic']:.1%}")
print(f"AUC vs. meaning changes:     NLI {top['auc_vs_hand_meaning']:.3f}  |  QQP {QQP_AUC:.3f}")
if GPT_COMPARED:
    print(f"Meaning changes NLI rejects but GPT-4o-mini accepts: {summary['meaning_changes_nli_rejects_gpt_accepts']}")
    print(f"Meaning changes GPT-4o-mini rejects but NLI accepts: {summary['meaning_changes_gpt_rejects_nli_accepts']}")
else:
    print("GPT-4o-mini comparison on the control pairs: skipped (no OR_API_KEY)")
print("\nNon-robust rate, all versions -> gated (random-drop expectation):")
for row in effect.itertuples():
    print(f"  {row.gate:<26} {row.model:<24} {row.nonrobust_rate_all:.3f} -> {row.nonrobust_rate_gated:.3f} ({row.nonrobust_rate_random_drop:.3f})")
print("\nFiles in nli_test/:", ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))